# Analysis 00
Set Up and Declare Files

In [6]:
# ParseEyeLinkAsc_script.ipynp
# Created 8/15/18 by DJ. & Modified by HH 05/23.
# https://github.com/\/ParseEyeLinkAscFiles/blob/master/ParseEyeLinkAsc.py

# Import packages
import os
import pandas as pd
import time
import numpy  as np
from ParseEyeLinkAsc import ParseEyeLinkAsc

# Declare filenames
DATA_DIR_INPUT = (r"~\Data\Eyetracking_00_Raw Data") # folder where the data sits
EL_Filename = '001_01.asc' # filename of the EyeLink file (.asc)
DATA_DIR = (r"~\Data\Eyetracking_01_Preprocessed Data")


# Create new folder if folder does not already exist: 
folder_name = EL_Filename[:3]

DATA_DIR_OUTPUT = os.path.join(DATA_DIR, folder_name)
if not os.path.exists(DATA_DIR_OUTPUT):
    os.mkdir(DATA_DIR_OUTPUT)

# Load Data into Pandas Dataframes

In [ ]:
# Navigate to data directory
os.chdir(DATA_DIR_INPUT)

# Load file in
dfTrial,dfMsg,dfFix,dfSacc,dfBlink,dfSamples = ParseEyeLinkAsc(EL_Filename)


In [ ]:
print(dfSamples[0:1000])
# print(dfSacc)
# print(dfMsg)


     tSample      LX     LY  LPupil  RX  RY  RPupil
0    4474186   700.2  713.8   224.0 NaN NaN     NaN
1    4474188   700.7  714.1   224.0 NaN NaN     NaN
2    4474190   701.2  714.1   223.0 NaN NaN     NaN
3    4474192   701.3  714.0   223.0 NaN NaN     NaN
4    4474194   701.5  713.8   223.0 NaN NaN     NaN
..       ...     ...    ...     ...  ..  ..     ...
995  4476176  1079.2  552.4   270.0 NaN NaN     NaN
996  4476178  1068.2  550.5   270.0 NaN NaN     NaN
997  4476180  1057.5  548.2   271.0 NaN NaN     NaN
998  4476182  1047.2  546.2   271.0 NaN NaN     NaN
999  4476184  1040.0  544.2   272.0 NaN NaN     NaN

[1000 rows x 7 columns]


# Save Results (Raw) to Excel Files
Helpful if you want to analyze them in another language or just save them for later.

In [ ]:
print('Saving results...')
t = time.time()
# Get file prefix from original filename
EL_FileStart = os.path.splitext(EL_Filename)[0]
print(EL_FileStart)

# Make master list of dataframes to write
allDataFrames = [dfTrial,dfMsg,dfFix,dfSacc,dfBlink] # the dataframes
allNames = ['Trial','Message','Fixation','Saccade','Blink'] # what they're called

# Write dataframes to .xlsx files
for i in range(len(allNames)):
    outFilename = '%s\%s_%s.xlsx'%(DATA_DIR_OUTPUT,EL_FileStart,allNames[i])
    print('   Saving %s output as %s...'%(allNames[i],outFilename))
    allDataFrames[i].to_excel(outFilename,float_format='%.1f',index=False)

# Save Samples to CSV file (too large for Excel)
outFilename_samples = '%s\%s_Samples.csv'%(DATA_DIR_OUTPUT,EL_FileStart)
print('   Saving Samples output as %s...'%(outFilename_samples))
dfSamples.to_csv(outFilename_samples)

print('Done! Took %f seconds.'%(time.time()-t))

Extract Movie Information from MSG

In [4]:
if file.endswith('01.asc'):
    #Extract MSG details that pertain to the Movies
    dfcontain_movies = dfMsg[dfMsg['text'].str.contains('movie')]
    dfcontain_movies = dfcontain_movies.astype({"text":str})

    # Adjust columns, clean up
    dfcontain_movies['movie'] = dfcontain_movies['text'].str[-20:]
    dfcontain_movies['val'] = dfcontain_movies['text'].str[:4]
    dfcontain_movies['movie'] = dfcontain_movies['movie'].str.replace('mp4.', '', regex=False)
    dfcontain_movies = dfcontain_movies.drop(['text'], axis=1)

    # ===== CLEAN UP MOVIE TIME STAMPS FILE  ===== #
    # locate the rows in the 'movie' column that contain the desired string, 
    # extract the corresponding time stamp from the 'time' column

    t_start = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('started'), 'time']
    dft_start = t_start.to_frame()
    dft_start.columns = ['t_start(ms)']                                       # Create new Column Name
    dft_start = dft_start.reset_index(drop=True)                              # Drop Index Row

    t_end = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('stopped'), 'time']
    dft_end = t_end.to_frame()
    dft_end.columns = ['t_end(ms)']
    dft_end = dft_end.reset_index(drop=True)

    movie_num = dfcontain_movies['movie'].loc[::2]
    movie_num  = movie_num.str[:8]
    dfmovie_num = movie_num.to_frame()
    dfmovie_num.columns = ['movie_num']
    dfmovie_num = dfmovie_num.reset_index(drop=True)

    dft_diff = pd.DataFrame(columns=['t_diff(ms)'], index=range(len(t_start)))
    dft_diff['t_diff(ms)'] = dft_end['t_end(ms)'] - dft_start['t_start(ms)']
    # print(dft_diff.head())

    dfhorizontal_stack = pd.concat([dft_start, dft_end, dfmovie_num, dft_diff], ignore_index=False, axis=1)
    dfhorizontal_stack = dfhorizontal_stack.reset_index(drop=True)
    # print(dfhorizontal_stack.head())

    outFilename_movies = '%s\%s_Movie_Timestamps.xlsx'%(DATA_OUTPUT, EL_FileStart)
    dfhorizontal_stack.to_excel(outFilename_movies)

if file.endswith('02.asc'):
    #Extract MSG details that pertain to the screenshots
    dfcontain_screenshots = dfMsg[dfMsg['text'].str.contains('screenshots')]
    dfcontain_screenshots = dfcontain_screenshots.astype({"text":str})

    # Adjust columns, clean up
    dfcontain_screenshots['screenshot'] = dfcontain_screenshots['text'].str[-20:]
    dfcontain_screenshots['val'] = dfcontain_movies['text'].str[:4]
    dfcontain_screenshots['screenshot'] = dfcontain_movies['screenshot'].str.replace('mp4.', '', regex=False)
    dfcontain_screenshots = dfcontain_screenshots.drop(['text'], axis=1)

    # ===== CLEAN UP MOVIE TIME STAMPS FILE  ===== #
    # locate the rows in the 'movie' column that contain the desired string, 
    # extract the corresponding time stamp from the 'time' column

    t_start = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('started'), 'time']
    dft_start = t_start.to_frame()
    dft_start.columns = ['t_start(ms)']                                       # Create new Column Name
    dft_start = dft_start.reset_index(drop=True)                              # Drop Index Row

    t_end = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('stopped'), 'time']
    dft_end = t_end.to_frame()
    dft_end.columns = ['t_end(ms)']
    dft_end = dft_end.reset_index(drop=True)

    movie_num = dfcontain_movies['movie'].loc[::2]
    movie_num  = movie_num.str[:8]
    dfmovie_num = movie_num.to_frame()
    dfmovie_num.columns = ['movie_num']
    dfmovie_num = dfmovie_num.reset_index(drop=True)

    dft_diff = pd.DataFrame(columns=['t_diff(ms)'], index=range(len(t_start)))
    dft_diff['t_diff(ms)'] = dft_end['t_end(ms)'] - dft_start['t_start(ms)']
    # print(dft_diff.head())

    dfhorizontal_stack = pd.concat([dft_start, dft_end, dfmovie_num, dft_diff], ignore_index=False, axis=1)
    dfhorizontal_stack = dfhorizontal_stack.reset_index(drop=True)
    # print(dfhorizontal_stack.head())

    outFilename_movies = '%s\%s_Movie_Timestamps.xlsx'%(DATA_OUTPUT, EL_FileStart)
    dfhorizontal_stack.to_excel(outFilename_movies)



Matrix Manipulation: Movie Information

In [ ]:
# locate the rows in the 'movie' column that contain the desired string, 
# extract the corresponding time stamp from the 'time' column

t_start = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('started'), 'time']
dft_start = t_start.to_frame()
dft_start.columns = ['t_start(ms)']                                       # Create new Column Name
dft_start = dft_start.reset_index(drop=True)                              # Drop Index Row

t_end = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('stopped'), 'time']
dft_end = t_end.to_frame()
dft_end.columns = ['t_end(ms)']
dft_end = dft_end.reset_index(drop=True)   

movie_num = dfcontain_movies['movie'].loc[::2]
movie_num  = movie_num.str[:8]
dfmovie_num = movie_num.to_frame()
dfmovie_num.columns = ['movie_num']
dfmovie_num = dfmovie_num.reset_index(drop=True)

dft_diff = pd.DataFrame(columns=['t_diff(ms)'], index=range(len(t_start)))
dft_diff['t_diff(ms)'] = dft_end['t_end(ms)'] - dft_start['t_start(ms)']
# print(dft_diff.head())

dfhorizontal_stack = pd.concat([dft_start, dft_end, dfmovie_num, dft_diff], ignore_index=False, axis=1)
dfhorizontal_stack = dfhorizontal_stack.reset_index(drop=True)
# print(dfhorizontal_stack.head())

outFilename_movies = '%s\%s_Movie_Timestamps.xlsx'%(DATA_DIR_OUTPUT,EL_Filename)
dfhorizontal_stack.to_excel(outFilename_movies)